# calibration-repro: train CIFAR-100 ResNets on Kaggle / Colab

Reproduces the training side of Guo et al. (2017), *On Calibration of Modern Neural Networks*.

**Kaggle:** Settings -> Accelerator *GPU T4 x2* (or P100), Internet *on*, then *Save Version -> Save & Run All* so the
run survives closing the browser (12 h limit per session). **Colab:** Runtime -> Change runtime type -> GPU.

Pick the runs in the `JOBS` cell (17 runs in total). Each run writes `results/logits/<model>_wd<wd>_seed<s>.npz`; finished runs are
skipped, so you can split the work over several sessions and merge the zips locally. Estimated times (not measured):
ResNet-110 ~3 h, ResNet-56 ~1.5 h per run on one T4.

In [ ]:
import os, subprocess
REPO = "https://github.com/BenarmasNADIR/calibration-repro.git"
if not os.path.exists("calibration-repro"):
    subprocess.run(["git", "clone", "--depth", "1", REPO], check=True)
os.chdir("calibration-repro")
print(os.getcwd())

In [ ]:
# Install the pinned requirements, except torch/torchvision and their CUDA wheels (nvidia-*, cuda-*, triton):
# keep the CUDA build of PyTorch that Kaggle/Colab ship, which matches their drivers.
!grep -vE '^(torch|torchvision|triton|nvidia-[a-z0-9-]+|cuda-[a-z-]+)==' requirements.txt > /tmp/req_no_torch.txt
!grep -cE '^[a-z]' /tmp/req_no_torch.txt
!pip install -q -r /tmp/req_no_torch.txt
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Optional sanity check (~1 min): CPU smoke test + unit tests.
!pip install -q pytest
!python src/train.py --config configs/smoke.yaml --device cpu
!python -m pytest -q

In [ ]:
# Choose what to run: one job = one training run (one line of arguments to src/train.py).
# Comment out lines to split the work across sessions; finished runs (existing .npz) are skipped anyway.
WDS = ["1e-5", "1e-4", "5e-4", "1e-3", "5e-3"]
JOBS = []
JOBS += [f"--config configs/resnet110_main.yaml --seed {s}" for s in [0, 1, 2]]                  # main result
JOBS += [f"--config configs/wd_ablation.yaml --wd {wd} --seed {s}" for s in [0, 1] for wd in WDS]  # ResNet-56
JOBS += [f"--config configs/depth_ablation.yaml --depth {d} --seed {s}" for s in [0, 1] for d in [20, 32]]
# Depth ablation at depth 56/110 reuses the wd=1e-4 runs above (same recipe and seeds).
print(len(JOBS), "jobs")

In [ ]:
# Run the jobs, one per visible GPU in parallel (Kaggle T4 x2 -> 2 at a time). Logs in results/logs/.
import torch, time, pathlib
pathlib.Path("results/logs").mkdir(parents=True, exist_ok=True)
n_gpu = max(torch.cuda.device_count(), 1)
queue, running = list(JOBS), {}
while queue or running:
    for gpu in range(n_gpu):
        if gpu not in running and queue:
            job = queue.pop(0)
            log = open("results/logs/" + job.replace("--", "").replace("configs/", "").replace(" ", "_") + ".log", "a")
            env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu))
            running[gpu] = (subprocess.Popen(["python", "src/train.py", *job.split()],
                                             stdout=log, stderr=subprocess.STDOUT, env=env), job)
            print(f"GPU {gpu}: started {job}")
    for gpu, (p, job) in list(running.items()):
        if p.poll() is not None:
            print(f"GPU {gpu}: finished {job} (exit {p.returncode})")
            del running[gpu]
    time.sleep(30)
!tail -n 2 results/logs/*.log

In [ ]:
# Zip logits (+ logs) for download. Kaggle: Output tab. Colab: the download below.
!cd results && zip -qr ../../logits.zip logits logs && ls -la ../../logits.zip
try:
    from google.colab import files
    files.download("../logits.zip")
except ImportError:
    pass

Then locally: unzip into `results/` (so files land in `results/logits/`) and run
`uv run python src/analyze.py`.